# Submission v6 — HRV Features + Relative Features + XGBoost + SMOTE

## What changed from v4/v5 (and why)

Research on subject-independent stress detection (WESAD, SWELL-KW benchmarks) shows
that generic models plateau at ~42% balanced accuracy using raw signal statistics —
exactly our LOPO ceiling. The key improvement is **proper HRV (Heart Rate Variability) features**:

| Feature | Why it generalises across people |
|---------|----------------------------------|
| SDNN | Overall HRV — high in baseline, low in stress — relative, not absolute |
| RMSSD | Short-term HRV — parasympathetic activity marker |
| pNN25 / pNN50 | % successive RR intervals >25/50ms — robust ratio feature |
| LF / HF / LF:HF | Sympathovagal balance — LF dominates during stress across ALL people |
| CV_RR | Coefficient of variation of RR — scale-invariant |

These are derived from **inter-beat intervals (RR = 60000/BPM)** computed from the 1Hz HR signal.
They are **ratio and relative** features — no person-specific baseline needed.

Also added:
- **EDA rate-of-change**: first derivative of EDA (SCR onset is a reliable stress cue)
- **EDA above-baseline ratio**: % of window where EDA > person's median EDA
- **Ratio features**: EDA/temp, HR/temp — cross-sensor ratios are person-invariant


In [1]:
%pip install xgboost scikit-learn pandas numpy scipy imbalanced-learn -q


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from scipy import stats as spstats
from scipy import signal as scsig
from sklearn.model_selection import StratifiedKFold, LeaveOneGroupOut
from sklearn.metrics import balanced_accuracy_score
from sklearn.impute import SimpleImputer
import xgboost as xgb
from imblearn.over_sampling import SMOTE
from collections import Counter

TRAIN_DATA  = pd.read_csv('train-sensor.csv')
TRAIN_LABEL = pd.read_csv('train-label.csv')
TEST_DATA   = pd.read_csv('test-sensor.csv')
TEST_LABEL  = pd.read_csv('test-label.csv')

TRAIN_LABEL['timestamp'] = TRAIN_LABEL['timestamp'].astype(float)
TEST_LABEL['timestamp']  = TEST_LABEL['timestamp'].astype(str).str.strip().astype(float)

print('TRAIN_DATA shape :', TRAIN_DATA.shape)
print('TRAIN_LABEL shape:', TRAIN_LABEL.shape)
print('TEST_DATA shape  :', TEST_DATA.shape)
print('TEST_LABEL shape :', TEST_LABEL.shape)
print()
print('Stress distribution:')
print(TRAIN_LABEL['stress'].value_counts().sort_index())


TRAIN_DATA shape : (4694400, 8)
TRAIN_LABEL shape: (815, 4)
TEST_DATA shape  : (5921280, 8)
TEST_LABEL shape : (1028, 4)

Stress distribution:
stress
0.0    162
1.0     66
2.0    587
Name: count, dtype: int64


In [3]:
SENSOR_COLS = ['accel_x','accel_y','accel_z','eda','heart_rate','temperature']
WINDOW_MS  = 180_000
HALF_MS    =  90_000
THIRD_MS   =  60_000

# ── Per-person stats for z-score normalisation ────────────────────────────────
def compute_pid_stats(sensor_df):
    d = {}
    for pid, grp in sensor_df.groupby('pid'):
        d[pid] = {}
        for c in SENSOR_COLS:
            v = grp[c].dropna().values.astype(float)
            d[pid][c+'_mu'] = float(np.mean(v)) if len(v)>0 else 0.0
            d[pid][c+'_sd'] = max(float(np.std(v)) if len(v)>0 else 1.0, 1e-6)
            d[pid][c+'_med'] = float(np.median(v)) if len(v)>0 else 0.0
    return d

train_pid_stats = compute_pid_stats(TRAIN_DATA)
test_pid_stats  = compute_pid_stats(TEST_DATA)
global_mu  = {c: np.median([train_pid_stats[p][c+'_mu']  for p in train_pid_stats]) for c in SENSOR_COLS}
global_sd  = {c: np.median([train_pid_stats[p][c+'_sd']  for p in train_pid_stats]) for c in SENSOR_COLS}
global_med = {c: np.median([train_pid_stats[p][c+'_med'] for p in train_pid_stats]) for c in SENSOR_COLS}

def znorm(col_series, mu, sd):
    v = col_series.dropna().values.astype(float)
    return (v - mu) / sd if len(v) > 0 else np.array([])

# ── HRV features from BPM signal ─────────────────────────────────────────────
def hrv_features(bpm_series):
    """
    Derive HRV features from instantaneous BPM.
    HR is sampled at 32Hz but updates every ~1s, so downsample to 1Hz first.
    RR(ms) = 60000 / BPM
    """
    feat = {}
    bpm = bpm_series.dropna().values.astype(float)
    if len(bpm) < 10:
        for k in ['sdnn','rmssd','pnn25','pnn50','mean_rr','cv_rr','lf_hf','lf_nu','hf_nu']:
            feat['hrv_' + k] = np.nan
        return feat

    # Downsample: keep one value per second (BPM updates ~1Hz)
    bpm_1hz = bpm[::32] if len(bpm) >= 32 else bpm
    if len(bpm_1hz) < 5:
        bpm_1hz = bpm

    rr = 60000.0 / np.clip(bpm_1hz, 30, 220)   # clip unrealistic BPM
    rr_diff = np.diff(rr)

    feat['hrv_sdnn']    = float(np.std(rr))
    feat['hrv_rmssd']   = float(np.sqrt(np.mean(rr_diff**2))) if len(rr_diff)>0 else 0.0
    feat['hrv_pnn25']   = float(np.mean(np.abs(rr_diff) > 25)) * 100 if len(rr_diff)>0 else 0.0
    feat['hrv_pnn50']   = float(np.mean(np.abs(rr_diff) > 50)) * 100 if len(rr_diff)>0 else 0.0
    feat['hrv_mean_rr'] = float(np.mean(rr))
    feat['hrv_cv_rr']   = feat['hrv_sdnn'] / feat['hrv_mean_rr'] if feat['hrv_mean_rr'] > 1e-6 else 0.0

    # Frequency domain (Welch PSD on RR sequence at 1Hz)
    try:
        nperseg = min(len(rr), 64)
        freqs, psd = scsig.welch(rr, fs=1.0, nperseg=nperseg)
        lf_mask = (freqs >= 0.04) & (freqs < 0.15)
        hf_mask = (freqs >= 0.15) & (freqs < 0.40)
        lf = float(np.trapezoid(psd[lf_mask], freqs[lf_mask])) if lf_mask.sum()>0 else 0.0
        hf = float(np.trapezoid(psd[hf_mask], freqs[hf_mask])) if hf_mask.sum()>0 else 0.0
        total = lf + hf
        feat['hrv_lf_hf']  = lf / hf     if hf > 1e-6 else 0.0
        feat['hrv_lf_nu']  = lf / total  if total > 1e-6 else 0.0
        feat['hrv_hf_nu']  = hf / total  if total > 1e-6 else 0.0
    except Exception:
        feat['hrv_lf_hf'] = feat['hrv_lf_nu'] = feat['hrv_hf_nu'] = np.nan

    return feat

print('HRV function defined.')
print('Test on P4DZ window:')
p = TRAIN_DATA[TRAIN_DATA['pid']=='P4DZ'].sort_values('timestamp')
lrow = TRAIN_LABEL[TRAIN_LABEL['pid']=='P4DZ'].iloc[50]
win = p[(p['timestamp'] >= float(lrow['timestamp'])-180000) &
        (p['timestamp'] <= float(lrow['timestamp']))]
print(hrv_features(win['heart_rate']))


HRV function defined.
Test on P4DZ window:
{'hrv_sdnn': 103.89888855207268, 'hrv_rmssd': 4.160769614924178, 'hrv_pnn25': 0.0, 'hrv_pnn50': 0.0, 'hrv_mean_rr': 799.9928438823116, 'hrv_cv_rr': 0.12987477243903625, 'hrv_lf_hf': 141.0207687913385, 'hrv_lf_nu': 0.9929587763218686, 'hrv_hf_nu': 0.0070412236781314174}


In [4]:
def extract_features(label_df, sensor_df, pid_stats):
    sensor_by_pid = {pid: grp.sort_values('timestamp').reset_index(drop=True)
                     for pid, grp in sensor_df.groupby('pid')}
    rows = []
    for _, lrow in label_df.iterrows():
        pid = lrow['pid']; ts = float(lrow['timestamp']); lid = lrow['id']
        feat = {'id': lid}
        sg = sensor_by_pid.get(pid)
        if sg is None: rows.append(feat); continue
        ta = sg['timestamp'].values
        wa  = sg.loc[(ta>=ts-WINDOW_MS)&(ta<=ts),          SENSOR_COLS]
        wf  = sg.loc[(ta>=ts-WINDOW_MS)&(ta<ts-HALF_MS),   SENSOR_COLS]
        wl  = sg.loc[(ta>=ts-HALF_MS)  &(ta<=ts),          SENSOR_COLS]
        wt1 = sg.loc[(ta>=ts-WINDOW_MS)&(ta<ts-2*THIRD_MS),SENSOR_COLS]
        wt3 = sg.loc[(ta>=ts-THIRD_MS) &(ta<=ts),          SENSOR_COLS]

        # ── Z-scored statistical features (from v2/v4) ───────────────────────
        for col in SENSOR_COLS:
            mu  = pid_stats.get(pid,{}).get(col+'_mu',  global_mu[col])
            sd  = pid_stats.get(pid,{}).get(col+'_sd',  global_sd[col])
            med = pid_stats.get(pid,{}).get(col+'_med', global_med[col])
            va  = znorm(wa[col],  mu, sd)
            vf  = znorm(wf[col],  mu, sd)
            vl  = znorm(wl[col],  mu, sd)
            vt1 = znorm(wt1[col], mu, sd)
            vt3 = znorm(wt3[col], mu, sd)
            p   = col
            if len(va)==0:
                for s in ['mean','std','min','max','median','skew','kurt','range',
                          'q25','q75','iqr','delta','slope','t1_mean','t3_mean',
                          't3t1_delta','cv','above_baseline_ratio']:
                    feat[f'{p}_{s}'] = np.nan
                continue
            feat[f'{p}_mean']   = float(np.mean(va));  feat[f'{p}_std']  = float(np.std(va))
            feat[f'{p}_min']    = float(np.min(va));   feat[f'{p}_max']  = float(np.max(va))
            feat[f'{p}_median'] = float(np.median(va))
            feat[f'{p}_skew']   = float(spstats.skew(va))     if len(va)>2 else 0.0
            feat[f'{p}_kurt']   = float(spstats.kurtosis(va)) if len(va)>2 else 0.0
            feat[f'{p}_range']  = float(np.max(va)-np.min(va))
            feat[f'{p}_q25']    = float(np.percentile(va,25))
            feat[f'{p}_q75']    = float(np.percentile(va,75))
            feat[f'{p}_iqr']    = float(np.percentile(va,75)-np.percentile(va,25))
            feat[f'{p}_delta']  = float(np.mean(vl)-np.mean(vf)) if len(vf)>0 and len(vl)>0 else 0.0
            feat[f'{p}_slope']  = float(np.polyfit(np.linspace(0,1,len(va)),va,1)[0]) if len(va)>2 else 0.0
            feat[f'{p}_t1_mean']    = float(np.mean(vt1)) if len(vt1)>0 else 0.0
            feat[f'{p}_t3_mean']    = float(np.mean(vt3)) if len(vt3)>0 else 0.0
            feat[f'{p}_t3t1_delta'] = feat[f'{p}_t3_mean'] - feat[f'{p}_t1_mean']
            rv = wa[col].dropna().values.astype(float); rm = np.mean(rv)
            feat[f'{p}_cv'] = float(np.std(rv)/abs(rm)) if abs(rm)>1e-6 else 0.0
            # % of window where signal is above person's global median (person-invariant ratio)
            feat[f'{p}_above_baseline_ratio'] = float(np.mean(rv > med))

        # ── HRV features (9 features, person-invariant) ──────────────────────
        feat.update(hrv_features(wa['heart_rate']))

        # ── EDA rate-of-change (SCR onset marker) ────────────────────────────
        eda_raw = wa['eda'].dropna().values.astype(float)
        if len(eda_raw) > 2:
            eda_diff = np.diff(eda_raw)
            feat['eda_roc_mean']     = float(np.mean(eda_diff))
            feat['eda_roc_max']      = float(np.max(eda_diff))
            feat['eda_roc_pos_ratio']= float(np.mean(eda_diff > 0))   # % time EDA rising
        else:
            feat['eda_roc_mean'] = feat['eda_roc_max'] = feat['eda_roc_pos_ratio'] = np.nan

        # ── Cross-sensor ratio features (person-invariant) ───────────────────
        eda_m  = wa['eda'].dropna().mean()
        hr_m   = wa['heart_rate'].dropna().mean()
        temp_m = wa['temperature'].dropna().mean()
        feat['ratio_eda_hr']   = eda_m / hr_m   if hr_m   > 1e-6 else 0.0
        feat['ratio_eda_temp'] = eda_m / temp_m if temp_m > 1e-6 else 0.0
        feat['ratio_hr_temp']  = hr_m  / temp_m if temp_m > 1e-6 else 0.0

        # ── Accel magnitude ──────────────────────────────────────────────────
        ax=wa['accel_x'].values; ay=wa['accel_y'].values; az=wa['accel_z'].values
        if len(ax)>0:
            mag=np.sqrt(ax**2+ay**2+az**2)
            feat['accel_mag_mean']=float(np.mean(mag))
            feat['accel_mag_std'] =float(np.std(mag))
            feat['accel_mag_max'] =float(np.max(mag))
        else:
            feat['accel_mag_mean']=feat['accel_mag_std']=feat['accel_mag_max']=np.nan

        # ── EDA x HR product (sympathetic co-activation) ─────────────────────
        emu=pid_stats.get(pid,{}).get('eda_mu',global_mu['eda'])
        esd=pid_stats.get(pid,{}).get('eda_sd',global_sd['eda'])
        hmu=pid_stats.get(pid,{}).get('heart_rate_mu',global_mu['heart_rate'])
        hsd=pid_stats.get(pid,{}).get('heart_rate_sd',global_sd['heart_rate'])
        ez=znorm(wa['eda'],emu,esd); hz=znorm(wa['heart_rate'],hmu,hsd)
        if len(ez)>2 and len(hz)>2:
            n=min(len(ez),len(hz))
            feat['eda_hr_product']=float(np.mean(ez[:n]*hz[:n]))
            feat['eda_hr_corr']   =float(np.corrcoef(ez[:n],hz[:n])[0,1])
        else:
            feat['eda_hr_product']=feat['eda_hr_corr']=0.0

        rows.append(feat)
    return pd.DataFrame(rows).set_index('id')

print('Extracting train features...')
train_features = extract_features(TRAIN_LABEL, TRAIN_DATA, train_pid_stats)
print(f'  shape: {train_features.shape}')
print('Extracting test features...')
test_features = extract_features(TEST_LABEL, TEST_DATA, test_pid_stats)
print(f'  shape: {test_features.shape}')


Extracting train features...
  shape: (815, 128)
Extracting test features...
  shape: (1028, 128)


In [5]:
tli    = TRAIN_LABEL.set_index('id')
y      = tli['stress'].astype(int)
groups = tli['pid']

imputer    = SimpleImputer(strategy='median')
X_imp      = pd.DataFrame(imputer.fit_transform(train_features),
                           columns=train_features.columns, index=train_features.index)
X_test_imp = pd.DataFrame(imputer.transform(test_features),
                           columns=test_features.columns, index=test_features.index)

counts        = Counter(y)
total         = len(y)
n_cls         = len(counts)
class_weights = {c: total/(n_cls*cnt) for c,cnt in counts.items()}
sample_weights = np.array([class_weights[yi] for yi in y])

def make_smote(X_tr, y_tr, seed=42):
    cnt1 = Counter(y_tr).get(1, 0)
    if cnt1 < 2: return X_tr, y_tr
    k = max(1, min(5, cnt1-1))
    return SMOTE(k_neighbors=k, random_state=seed).fit_resample(X_tr, y_tr)

def get_sw(y_arr):
    return np.array([class_weights[int(yi)] for yi in y_arr])

print('X_imp shape:', X_imp.shape)
print('New features vs v4 (107):', X_imp.shape[1] - 107, 'additional')
print('Class weights:', {k: round(v,3) for k,v in class_weights.items()})


X_imp shape: (815, 128)
New features vs v4 (107): 21 additional
Class weights: {1: 4.116, 0: 1.677, 2: 0.463}


In [6]:
# XGBoost >= 2.0: early_stopping_rounds in constructor, NOT in .fit()
XGB_PARAMS = dict(
    n_estimators          = 800,
    learning_rate         = 0.02,
    max_depth             = 5,
    subsample             = 0.7,
    colsample_bytree      = 0.7,
    reg_alpha             = 0.3,
    reg_lambda            = 1.0,
    min_child_weight      = 3,
    early_stopping_rounds = 50,
    objective             = 'multi:softprob',
    num_class             = 3,
    eval_metric           = 'mlogloss',
    n_jobs                = -1,
    verbosity             = 0,
)

print('=== LOPO CV — XGBoost + HRV features + SMOTE ===')
logo = LeaveOneGroupOut()
lopo_scores = []

for tr_idx, val_idx in logo.split(X_imp, y, groups):
    pid_val = groups.iloc[val_idx[0]]
    y_val   = y.iloc[val_idx]
    if len(y_val.unique()) < 2:
        print(f'  Skip {pid_val}: only 1 class'); continue

    X_tr = X_imp.iloc[tr_idx].values
    y_tr = y.iloc[tr_idx].values
    X_va = X_imp.iloc[val_idx].values

    X_sm, y_sm = make_smote(X_tr, y_tr)
    sw_sm = get_sw(y_sm)

    m = xgb.XGBClassifier(**{**XGB_PARAMS, 'random_state': 42})
    m.fit(X_sm, y_sm,
          sample_weight = sw_sm,
          eval_set      = [(X_va, y_val)],
          verbose       = False)

    sc = balanced_accuracy_score(y_val, m.predict(X_va))
    print(f'  Leave out {pid_val}: {sc:.4f}  (n={len(val_idx)}, classes={sorted(y_val.unique())})')
    lopo_scores.append(sc)

print(f'\nv6 XGBoost+HRV LOPO = {np.mean(lopo_scores):.4f} +/- {np.std(lopo_scores):.4f}')
print('v4 reference         = 0.4288  (previous best)')
print('Improvement          =', round(np.mean(lopo_scores) - 0.4288, 4))


=== LOPO CV — XGBoost + HRV features + SMOTE ===
  Leave out 43JW: 0.1868  (n=93, classes=[np.int64(0), np.int64(2)])
  Leave out C8Q6: 0.4718  (n=152, classes=[np.int64(0), np.int64(2)])
  Leave out DT5C: 0.3488  (n=90, classes=[np.int64(0), np.int64(1), np.int64(2)])
  Leave out F1ZM: 0.4478  (n=137, classes=[np.int64(1), np.int64(2)])
  Leave out HDS9: 0.6688  (n=135, classes=[np.int64(0), np.int64(2)])
  Leave out P4DZ: 0.3396  (n=144, classes=[np.int64(0), np.int64(1), np.int64(2)])
  Leave out TPQI: 0.4607  (n=64, classes=[np.int64(0), np.int64(2)])

v6 XGBoost+HRV LOPO = 0.4178 +/- 0.1378
v4 reference         = 0.4288  (previous best)
Improvement          = -0.011


In [7]:
print('=== Final ensemble: XGBoost + HRV, 3 seeds x 5 folds ===')
SEEDS = [42, 7, 123]
all_test_proba = []
all_cv_scores  = []

for seed in SEEDS:
    skf        = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    seed_proba = np.zeros((len(X_test_imp), 3))
    fold_scores = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_imp, y)):
        X_tr = X_imp.iloc[tr_idx].values
        y_tr = y.iloc[tr_idx].values
        X_va = X_imp.iloc[val_idx].values
        y_va = y.iloc[val_idx]

        X_sm, y_sm = make_smote(X_tr, y_tr, seed=seed)
        sw_sm = get_sw(y_sm)

        m = xgb.XGBClassifier(**{**XGB_PARAMS,
                                  'random_state': seed,
                                  'early_stopping_rounds': 80})
        m.fit(X_sm, y_sm,
              sample_weight = sw_sm,
              eval_set      = [(X_va, y_va)],
              verbose       = False)

        sc = balanced_accuracy_score(y_va, m.predict(X_va))
        fold_scores.append(sc)
        seed_proba += m.predict_proba(X_test_imp.values)
        print(f'  Seed {seed} Fold {fold+1}: val BA = {sc:.4f}')

    seed_proba /= 5
    all_test_proba.append(seed_proba)
    all_cv_scores.append(np.mean(fold_scores))
    print(f'  Seed {seed} mean CV = {np.mean(fold_scores):.4f}')

print(f'\nEnsemble CV = {np.mean(all_cv_scores):.4f}')

final_proba = np.mean(all_test_proba, axis=0)
final_preds = np.argmax(final_proba, axis=1).astype(int)

print('\nPrediction distribution:')
for u, c in zip(*np.unique(final_preds, return_counts=True)):
    print(f'  class {u}: {c}')


=== Final ensemble: XGBoost + HRV, 3 seeds x 5 folds ===
  Seed 42 Fold 1: val BA = 0.8176
  Seed 42 Fold 2: val BA = 0.7819
  Seed 42 Fold 3: val BA = 0.8925
  Seed 42 Fold 4: val BA = 0.8154
  Seed 42 Fold 5: val BA = 0.8552
  Seed 42 mean CV = 0.8325
  Seed 7 Fold 1: val BA = 0.7826
  Seed 7 Fold 2: val BA = 0.7962
  Seed 7 Fold 3: val BA = 0.8169
  Seed 7 Fold 4: val BA = 0.8267
  Seed 7 Fold 5: val BA = 0.7813
  Seed 7 mean CV = 0.8007
  Seed 123 Fold 1: val BA = 0.8471
  Seed 123 Fold 2: val BA = 0.7325
  Seed 123 Fold 3: val BA = 0.8837
  Seed 123 Fold 4: val BA = 0.8183
  Seed 123 Fold 5: val BA = 0.7943
  Seed 123 mean CV = 0.8152

Ensemble CV = 0.8161

Prediction distribution:
  class 0: 341
  class 1: 68
  class 2: 619


In [8]:
submission = pd.DataFrame({'id': TEST_LABEL['id'].values, 'stress': final_preds})
submission.to_csv('submission_v6_hrv.csv', index=False)
print('submission_v6_hrv.csv saved!')
print(submission.head(10))


submission_v6_hrv.csv saved!
     id  stress
0  1227       2
1  1228       2
2  1229       2
3  1230       2
4  1231       0
5  1232       2
6  1233       1
7  1234       2
8  1235       1
9  1236       2
